<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/VSCODE_LocalLLM_Ollama_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 라이브러리를 임포트한다.

In [ ]:
pip install langchain langchain-ollama pydantic openpyxl

모델을 GEMMA3로 설정한다.

In [ ]:
!ollama pull gemma3:4b

필요한 모듈을 임포트 한다.

In [ ]:
from pathlib import Path
from typing import Literal
from pydantic import BaseModel, Field
from openpyxl import load_workbook

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

기본 환경을 설정한다.

In [ ]:
ROOT = Path(r"C:\VirtualC") # 가상의 폴더공간
MODEL = "gemma3:4b"   # DeepMind 모델 Gemma 40억 매개변수

가상의 실습 드라이브 구성한다.

In [ ]:
# --------------------------------------------------
# 실습용 가상 C 드라이브 만들기
# --------------------------------------------------

(ROOT / "Reports").mkdir(parents=True, exist_ok=True)
(ROOT / "Finance").mkdir(parents=True, exist_ok=True)
(ROOT / "AI").mkdir(parents=True, exist_ok=True)

(ROOT / "Reports" / "AI_report.txt").write_text(
    "이 파일은 AI 도입 보고서입니다.",
    encoding="utf-8"
)

(ROOT / "Reports" / "meeting_memo.txt").write_text(
    "회의 메모입니다. 예산과 일정 논의가 있었습니다.",
    encoding="utf-8"
)

(ROOT / "AI" / "ollama_note.txt").write_text(
    "Ollama와 Gemma 모델을 이용한 로컬 LLM 실습 메모입니다.",
    encoding="utf-8"
)

wb = Workbook()
ws = wb.active
ws.title = "Sales"
ws.append(["항목", "내용", "금액"])
ws.append(["매출", "2026년 1분기 매출 데이터", 1000])
ws.append(["비용", "마케팅 비용", 300])
wb.save(ROOT / "Finance" / "sales_2026.xlsx")

wb = Workbook()
ws = wb.active
ws.title = "LLM"
ws.append(["모델", "설명"])
ws.append(["Gemma", "가벼운 로컬 LLM 실습용 모델"])
ws.append(["Qwen", "성능은 좋지만 무거운 모델"])
wb.save(ROOT / "AI" / "local_llm_models.xlsx")

실제 로컬 드라이브 검색 함수를 정의한다.

In [ ]:
def search_local_files(mode, keyword):
    results = []

    if mode == "filename":
        for file_path in ROOT.rglob("*"):
            if file_path.is_file():
                if keyword.lower() in file_path.name.lower():
                    results.append(
                        str(file_path.relative_to(ROOT))
                    )

    elif mode == "excel_content":
        for file_path in ROOT.rglob("*.xlsx"):
            wb = load_workbook(
                file_path,
                read_only=True,
                data_only=True
            )

            for sheet in wb.worksheets:
                for row in sheet.iter_rows():
                    for cell in row:
                        if cell.value is None:
                            continue

                        cell_text = str(cell.value)

                        if keyword.lower() in cell_text.lower():
                            results.append(
                                f"{file_path.relative_to(ROOT)} / {sheet.title}!{cell.coordinate} = {cell_text}"
                            )

            wb.close()

    return results

LLM 모델을 생성한다.

In [ ]:
llm = ChatOllama(
    model=MODEL,
    temperature=0
)

LLM에게 어떻게 대답해야 할지 양식을 지정한다.<BR>
with_structured_output() 함수에 전달할 내용이다.

In [ ]:
# 이 SearchPlan은  LLM에게 다음과 같이 지시하는 것과 같다.

#  너는 반드시 아래 두 칸만 채워라.
#
#   mode: filename 또는 excel_content 중 하나
#   keyword: 검색할 핵심 단어


class SearchPlan(BaseModel):
    mode: Literal["filename", "excel_content"] = Field(
        description="filename 또는 excel_content 중 하나"
    )
    keyword: str = Field(
        description="검색할 핵심 단어"
    )

LLM으로 검색 의도를 해석하도록 하는 프롬프트

In [ ]:
planner_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
너는 사용자의 한국어 파일 검색 요청을 검색 계획으로 바꾸는 분류기다.

mode는 두 가지다.

1. filename
- 파일 이름에 keyword가 들어간 파일을 찾는다.
- 예: "AI 들어가는 파일 찾아줘"

2. excel_content
- 엑셀 파일 내부 셀 값에 keyword가 쓰인 파일을 찾는다.
- 예: "매출 단어가 쓰인 엑셀 파일 찾아줘"

규칙:
- keyword에는 조사와 불필요한 말을 제거한다.
- "엑셀", "액셀", "xlsx", "단어가 쓰인", "내용에 들어간"이 있으면 excel_content로 본다.
- 단순히 "들어가는 파일", "이름에 들어간 파일"이면 filename으로 본다.
""".strip()
        ),
        (
            "human",
            "{user_query}"
        )
    ]
)

파이프라인을 구성한다.<BR>
with_structured_output함수는 일반 LLM을 SearchPlan 형식으로 답하는 LLM으로 바꾼다.

In [ ]:
planner_chain = planner_prompt | llm.with_structured_output(SearchPlan)

답변 chain을 출력해보자.

In [ ]:
display( planner_chain )

출력을 어떻게 할지 프롬프트로 지시한다.

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
너는 로컬 파일 검색 결과를 사용자에게 설명하는 AI다.
검색 결과에 없는 파일은 지어내지 마라.
짧고 명확하게 한국어로 답하라.
""".strip()
        ),
        (
            "human",
            """
사용자 요청:
{user_query}

검색 위치:
{root}

검색 계획:
{plan}

검색 결과:
{search_context}

위 검색 결과만 근거로 최종 답변을 작성하라.
""".strip()
        )
    ]
)

대답 체인을 구성한다.

In [ ]:
answer_chain = answer_prompt | llm

이제 루프를 돌며 명령을 받아 들인다.

In [ ]:
print("로컬 파일 검색 RAG 실습 시작")
print("검색 위치:", ROOT)
print("예시:")
print("- AI 들어가는 파일 찾아줘")
print("- 매출 단어가 쓰인 엑셀 파일 찾아줘")
print("- Gemma 단어가 쓰인 액셀 파일 찾아줘")
print("- exit")

while True:
    user_query = input("\n요청: ").strip()

    if user_query.lower() == "exit":
        print("종료합니다.")
        break

    plan = planner_chain.invoke(
        {
            "user_query": user_query
        }
    )

    print("\n[LLM 검색 의도 해석]")
    print(plan)

    results = search_local_files(
        plan.mode,
        plan.keyword
    )

    if len(results) == 0:
        search_context = "검색 결과 없음"
    else:
        search_context = "\n".join(results)

    print("\n[Python 검색 결과]")
    print(search_context)

    answer = answer_chain.invoke(
        {
            "user_query": user_query,
            "root": str(ROOT),
            "plan": plan,
            "search_context": search_context
        }
    )

    print("\n[LLM 최종 답변]")
    print(answer.content)